In [21]:

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")


TICKER      = "RELIANCE.NS"
START       = "2019-01-01"
END         = "2024-12-31"
LOOKBACK    = 20          
ENTRY_Z     = 1.5         
EXIT_Z      = 0.0        
CAPITAL     = 500_000     
RISK_FREE   = 0.065  

raw = yf.download(TICKER, start=START, end=END, progress=False)
df  = raw[["Close"]].copy()
df.columns = ["price"]
df.dropna(inplace=True)

def compute_signals(df: pd.DataFrame, lookback: int,
                    entry_z: float, exit_z: float) -> pd.DataFrame:
    
    
    
    df = df.copy()
    df["rolling_mean"] = df["price"].rolling(lookback).mean()
    df["rolling_std"]  = df["price"].rolling(lookback).std()
    df["zscore"]       = (df["price"] - df["rolling_mean"]) / df["rolling_std"]

    
    positions = np.zeros(len(df))
    pos = 0
    for i in range(lookback, len(df)):
        z = df["zscore"].iloc[i]
        if pos == 0:
            if z < -entry_z:
                pos = 1          # go long
            elif z > entry_z:
                pos = -1         # go short
        elif pos == 1 and z >= -exit_z:
            pos = 0              # close long
        elif pos == -1 and z <= exit_z:
            pos = 0              # close short
        positions[i] = pos

    df["position"]      = positions
    df["signal_change"] = df["position"].diff().fillna(0)
    return df.dropna(subset=["zscore"])

df = compute_signals(df, LOOKBACK, ENTRY_Z, EXIT_Z)
print(f"Total signal changes (trade entries/exits): {(df['signal_change'] != 0).sum()}")

def simulate_portfolio(df: pd.DataFrame, capital: float) -> pd.DataFrame:
    
    df = df.copy()
    df["daily_ret"]     = df["price"].pct_change()
    df["strat_ret"]     = df["position"].shift(1) * df["daily_ret"]
    df["bh_ret"]        = df["daily_ret"]

    df["strat_equity"]  = capital * (1 + df["strat_ret"].fillna(0)).cumprod()
    df["bh_equity"]     = capital * (1 + df["bh_ret"].fillna(0)).cumprod()

    # drawdown
    roll_max            = df["strat_equity"].cummax()
    df["drawdown_pct"]  = (df["strat_equity"] - roll_max) / roll_max * 100
    return df


df = simulate_portfolio(df, CAPITAL)

def compute_metrics(df: pd.DataFrame, capital: float,
                    risk_free: float = 0.065) -> dict:
    """
    Returns a dict of key performance metrics.
    """
    strat_ret  = df["strat_ret"].dropna()
    bh_ret     = df["bh_ret"].dropna()
    rf_daily   = risk_free / 252

    # CAGR
    years      = len(strat_ret) / 252
    end_val    = df["strat_equity"].iloc[-1]
    bh_end     = df["bh_equity"].iloc[-1]
    cagr       = (end_val / capital) ** (1 / years) - 1
    bh_cagr    = (bh_end / capital) ** (1 / years) - 1

    # Sharpe
    excess     = strat_ret - rf_daily
    sharpe     = (excess.mean() / excess.std()) * np.sqrt(252)

    # Sortino
    neg_ret    = strat_ret[strat_ret < rf_daily] - rf_daily
    sortino    = (excess.mean() / neg_ret.std()) * np.sqrt(252) if len(neg_ret) else np.nan

    # Calmar
    max_dd     = df["drawdown_pct"].min()
    calmar     = (cagr / abs(max_dd / 100)) if max_dd != 0 else np.nan

    # Win rate
    trades     = identify_trades(df)
    wins       = sum(1 for t in trades if t["pnl_pct"] > 0)
    win_rate   = wins / len(trades) * 100 if trades else 0

    # Volatility
    ann_vol    = strat_ret.std() * np.sqrt(252)

    return {
        "Strategy CAGR":      f"{cagr*100:.2f}%",
        "Buy & Hold CAGR":    f"{bh_cagr*100:.2f}%",
        "Sharpe Ratio":       f"{sharpe:.3f}",
        "Sortino Ratio":      f"{sortino:.3f}",
        "Calmar Ratio":       f"{calmar:.3f}",
        "Max Drawdown":       f"{max_dd:.2f}%",
        "Ann. Volatility":    f"{ann_vol*100:.2f}%",
        "Win Rate":           f"{win_rate:.1f}%",
        "Total Trades":       len(trades),
        "Final Equity (₹)":   f"₹{end_val:,.0f}",
        "B&H Equity (₹)":     f"₹{bh_end:,.0f}",
    }

def identify_trades(df: pd.DataFrame) -> list:
    """Extract individual completed trades from position series."""
    trades = []
    entry  = None
    for i in range(1, len(df)):
        chg = df["signal_change"].iloc[i]
        if entry is None and chg != 0:
            entry = {
                "date":  df.index[i],
                "price": df["price"].iloc[i],
                "dir":   int(df["position"].iloc[i]),
            }
        elif entry is not None and df["position"].iloc[i] == 0:
            exit_p   = df["price"].iloc[i]
            pnl_pct  = entry["dir"] * (exit_p - entry["price"]) / entry["price"] * 100
            trades.append({
                "entry_date":  entry["date"],
                "exit_date":   df.index[i],
                "direction":   "Long" if entry["dir"] == 1 else "Short",
                "entry_price": entry["price"],
                "exit_price":  exit_p,
                "pnl_pct":     round(pnl_pct, 3),
            })
            entry = None
    return trades


metrics = compute_metrics(df, CAPITAL, RISK_FREE)
print("\n" + "=" * 45)
print("  PERFORMANCE METRICS — RELIANCE MEAN REVERSION")
print("=" * 45)
for k, v in metrics.items():
    print(f"  {k:<22} {v}")
print("=" * 45)

trades = identify_trades(df)
trades_df = pd.DataFrame(trades)
trades_df["pnl_pct"] = trades_df["pnl_pct"].apply(
    lambda x: f"+{x:.2f}%" if x >= 0 else f"{x:.2f}%"
)
print(f"\nTotal trades: {len(trades_df)}")
print(trades_df.tail(10).to_string(index=False))





Total signal changes (trade entries/exits): 145

  PERFORMANCE METRICS — RELIANCE MEAN REVERSION
  Strategy CAGR          -6.41%
  Buy & Hold CAGR        14.63%
  Sharpe Ratio           -0.388
  Sortino Ratio          -0.499
  Calmar Ratio           -0.101
  Max Drawdown           -63.45%
  Ann. Volatility        25.46%
  Win Rate               69.4%
  Total Trades           72
  Final Equity (₹)       ₹340,552
  B&H Equity (₹)         ₹1,102,752

Total trades: 72
entry_date  exit_date direction  entry_price  exit_price pnl_pct
2024-05-03 2024-05-21      Long  1423.464355 1425.573730  +0.15%
2024-05-23 2024-05-30     Short  1475.131958 1414.381470  +4.12%
2024-06-03 2024-06-04     Short  1499.228516 1387.009155  +7.49%
2024-06-26 2024-07-22     Short  1502.901367 1489.649536  +0.88%
2024-07-23 2024-08-19      Long  1476.968262 1482.479126  +0.37%
2024-08-29 2024-09-06     Short  1514.874756 1458.997803  +3.69%
2024-09-11 2024-09-23      Long  1445.725952 1487.434326  +2.88%
2024-09-27 